In [37]:
from pathlib import Path

p = Path("/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py")
backup = p.with_suffix(".py.bak")

# 백업
if not backup.exists():
    backup.write_text(p.read_text(encoding="utf-8"), encoding="utf-8")

lines = p.read_text(encoding="utf-8").splitlines(keepends=True)

new_lines = []
i = 0
patched_1 = False
patched_2 = False

while i < len(lines):
    s = lines[i].lstrip()

    if s.startswith("ExtLinkBracketedRegex = re.compile("):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + "ExtLinkBracketedRegex = re.compile(\n",
            indent + "    '\\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\\s*([^\\]\\x00-\\x08\\x0a-\\x1F]*?)\\]',\n",
            indent + "    re.S | re.U | re.I)\n",
        ])
        i += 3
        patched_1 = True
        continue

    if s.startswith("EXT_IMAGE_REGEX = re.compile("):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + "EXT_IMAGE_REGEX = re.compile(\n",
            indent + '    r"""^(http://|https://)([^][<>"\\x00-\\x20\\x7F\\s]+)\n',
            indent + '    /([A-Za-z0-9_.,~%\\-+&;#*?!=()@\\x80-\\xFF]+)\\.(gif|png|jpg|jpeg)$""",\n',
            indent + "    re.X | re.S | re.U | re.I)\n",
        ])
        i += 4
        patched_2 = True
        continue

    new_lines.append(lines[i])
    i += 1

p.write_text("".join(new_lines), encoding="utf-8")

print("backup:", backup)
print("patched ExtLinkBracketedRegex:", patched_1)
print("patched EXT_IMAGE_REGEX:", patched_2)
print("done")

backup: /usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py.bak
patched ExtLinkBracketedRegex: True
patched EXT_IMAGE_REGEX: True
done


In [38]:
!python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text

/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:33: SyntaxWarning: invalid escape sequence '\w'
  tailRE = re.compile('\w+')
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:171: SyntaxWarning: invalid escape sequence '\.'
  text = re.sub(u' (,:\.\)\]»)', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:172: SyntaxWarning: invalid escape sequence '\['
  text = re.sub(u'(\[\(«) ', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:379: SyntaxWarning: invalid escape sequence '\['
  '\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\s*([^\]\x00-\x08\x0a-\x1F]*?)\]',
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:733: SyntaxWarning: invalid escape sequence '\w'
  return re.sub("&#?(\w+);", fixup, text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:1278: SyntaxWarning: invalid escape sequence '\['
  reOpen = re.compile('{{2,}|\[{2,}')
/usr/local/lib/

In [39]:
# 코랩환경 기준

# 데이터 파싱 을 위한 패키지 설치
!pip install wikiextractor

In [40]:
# 위키피디아 데이터 다운로드, 전처리에서 사용할 형태소 분석기 (Mecab)설치

!git clone https://github.com//SOMJANG/Mecab-ko-for-Google-colab.github
%cd Mecab-ko-for-Google-colab.github
!bash install_mecab-ko_on_colab190912.sh


Cloning into 'Mecab-ko-for-Google-colab.github'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'Mecab-ko-for-Google-colab.github'
/content
bash: install_mecab-ko_on_colab190912.sh: No such file or directory


In [41]:
# 위키피디아 덤프(위키피디아 데이터) 다운로드
!wget https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2

--2026-03-10 05:47:46--  https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1259021343 (1.2G) [application/octet-stream]
Saving to: ‘kowiki-latest-pages-articles.xml.bz2.1’

kowiki-latest-pages 100%[===================>]   1.17G   839KB/s    in 29m 35s 

2026-03-10 06:17:21 (693 KB/s) - ‘kowiki-latest-pages-articles.xml.bz2.1’ saved [1259021343/1259021343]



In [42]:
# 위키익스트랙터 를 이용한 위키피디아 덤프 파싱
!python -m wikiextractor.wikiextractor kowiki-latest-pages-articles.xml.bz2

/usr/bin/python3: No module named wikiextractor.wikiextractor


In [43]:
# 현재 경로에 있는 디렉터리와 파일들 의 리스트 받아오기
%ls

kowiki-latest-pages-articles.xml.bz2    output_file.txt  text/
kowiki-latest-pages-articles.xml.bz2.1  sample_data/


In [44]:
# 운영체제 기능 사용
import os
# 정규표현
import re

In [45]:
import os
print(os.listdir("/content"))

['.config', 'kowiki-latest-pages-articles.xml.bz2', 'output_file.txt', 'text', 'kowiki-latest-pages-articles.xml.bz2.1', 'sample_data']


In [46]:
import os
import shutil

os.makedirs("/content/text", exist_ok=True)

shutil.copy(
    "/content/kowiki-latest-pages-articles.xml.bz2",
    "/content/text/kowiki-latest-pages-articles.xml.bz2"
)

print(os.listdir("/content/text"))

['AE', 'AF', 'AN', 'AC', 'AL', 'kowiki-latest-pages-articles.xml.bz2', 'AA', 'AB', 'AH', 'AM', 'AI', 'AD', 'AG', 'AJ', 'AK']


In [47]:
os.listdir('text')



['AE',
 'AF',
 'AN',
 'AC',
 'AL',
 'kowiki-latest-pages-articles.xml.bz2',
 'AA',
 'AB',
 'AH',
 'AM',
 'AI',
 'AD',
 'AG',
 'AJ',
 'AK']

In [48]:
# AA라는 디렉토리 파일 확인
# %ls text/AA

%ls
%ls text

kowiki-latest-pages-articles.xml.bz2    output_file.txt  text/
kowiki-latest-pages-articles.xml.bz2.1  sample_data/
AA/  AC/  AE/  AG/  AI/  AK/  AM/  kowiki-latest-pages-articles.xml.bz2
AB/  AD/  AF/  AH/  AJ/  AL/  AN/


In [49]:
!pip install wikiextractor

In [50]:
# !python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text


In [51]:
import bz2

file_path = "/content/kowiki-latest-pages-articles.xml.bz2"

with bz2.open(file_path, "rt", encoding="utf-8", errors="ignore") as f:
    for i in range(20):
        print(f.readline())

<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.11/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.11/ http://www.mediawiki.org/xml/export-0.11.xsd" version="0.11" xml:lang="ko">

  <siteinfo>

    <sitename>위키백과</sitename>

    <dbname>kowiki</dbname>

    <base>https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EB%8C%80%EB%AC%B8</base>

    <generator>MediaWiki 1.46.0-wmf.17</generator>

    <case>first-letter</case>

    <namespaces>

      <namespace key="-2" case="first-letter">미디어</namespace>

      <namespace key="-1" case="first-letter">특수</namespace>

      <namespace key="0" case="first-letter" />

      <namespace key="1" case="first-letter">토론</namespace>

      <namespace key="2" case="first-letter">사용자</namespace>

      <namespace key="3" case="first-letter">사용자토론</namespace>

      <namespace key="4" case="first-letter">위키백과</namespace>

      <namespace key="5" case="first-letter

반복되는 내용 확인

In [52]:
# AA- AF 디렉토리 안의 wiki 숫자 형태의 수많은 파일들을 하나로 통합하는 과정 진행
# AA~ AF 디렉토리 안 모든 파일들의 경로를 리스트 형태로 저장

import os
import re

def list_wiki(dirname):
    filepaths = []
    filenames = os.listdir(dirname)

    for filename in filenames:
        filepath = os.path.join(dirname, filename)

        if os.path.isdir(filepath):
            filepaths.extend(list_wiki(filepath))
        else:
            find = re.findall(r"wiki_[0-9][0-9]", filepath)
            if len(find) > 0:
                filepaths.append(filepath)

    return sorted(filepaths)

In [53]:
# 총 파일의 개수 확인
filepaths = list_wiki("text")
len(filepaths)

1354

In [54]:
# output_file.txt 에 850개 파일을 전부 합치기
with open("output_file.txt", "w") as outfile:
  for filename in filepaths:
    with open(filename) as infile:
      contents = infile.read()
      outfile.write(contents)


In [55]:
f = open('output_file.txt', encoding='utf-8')

i = 0

while True:
  line = f.readline()
  if line != '\n':
    i = i+1
    print("%d번째 줄 : " %i + line)
  if i == 10:
    break
# open 을 했다면 꼭 닫아주어야 한다.
f.close()

1번째 줄 : <doc id="5" url="https://ko.wikipedia.org/wiki?curid=5" title="지미 카터">

2번째 줄 : 지미 카터

3번째 줄 : 제임스 얼 "지미 카터 주니어"(, 1924년 10월 1일~2024년 12월 29일)는 미국의 제39대 대통령 (1977-81)을 지낸 미국의 정치인이다. 민주당 소속으로 1963년부터 1967년까지 조지아주 상원 의원, 1971년부터 1975년까지 조지아주의 76대 주지사을 지냈다. 카터는 100세까지 산 최초의 대통령으로 미국 역사상 가장 장수한 대통령이다.

4번째 줄 : 카터는 조지아주 플레인스에서 태어나고 자랐다. 1946년 미국 해군사관학교를 졸업하고 미국 해군 잠수함에 승선했다. 카터는 군 복무를 마치고 고향으로 돌아와 가족의 땅콩 재배 사업을 되살렸다. 카터는 인종 분리 정책에 반대하며 성장하던 민권 운동을 지지했고, 민주당 내에서 활동가가 되었다. 1963년부터 1967년까지 조지아주 상원 의원으로 재직하였고, 1971년부터 1975년까지 조지아 주지사로 재직했다. 조지아 주 밖에서는 잘 알려지지 않은 다크호스 후보였던 카터는 민주당 후보로 지명되어 1976년 대선에서 공화당의 현직 대통령인 제럴드 포드를 상대로 신승했다.

5번째 줄 : 카터는 취임 둘째 날 베트남 전쟁에서 병역을 기피한 모든 사람들을 사면했다. 에너지부와 교육부를 설립했으며, 에너지 절약, 가격 통제, 신기술을 포함한 국가 에너지 정책을 만들었다. 스태그플레이션에 대응하는 동시에 카터는 캠프 데이비드 협정, 파나마 운하 조약, 제2차 전략 무기 제한 협상을 성공적으로 추진했다. 하지만 임기 말에는 이란 인질 사태, 에너지 위기, 스리마일 섬 사고, 니카라과 혁명, 그리고 소련의 아프가니스탄 침공 등 위기가 이어졌다. 아프가니스탄 침공에 대응하여 카터는 데탕트 정책을 종식시키고 카터 독트린을 선포했으며, 소련에 곡물 금수조치를 부과하고, 1980년 모스크바 하계 올림픽에 대한 다국적 보이콧을 주

교안 59 페이지 참조

In [56]:
# 형태소 분석
from tqdm import tqdm
from konlpy.tag import Mecab

# Mecab을 사용한 토큰화 진행
mecab = Mecab()

# out_file 에는 총 몇줄이 있을까
f = open('output_file.txt', encoding='utf-8')
lines = f.read().splitlines()
print()

ModuleNotFoundError: No module named 'konlpy'

In [ ]:
# 상위 10 개만 출력
lines[:10]

In [ ]:
# 아무런 단어도 들어있지 않은 '' 와 같은 줄도 존재한다.
# 제외하고 형태소 분석을 수행한다.

result = []

for line in tqdm(lines):
  # 빈 문자열이 아닌 경우에만 수행
  if line:
    result.append(mecab.morphs(line))

In [ ]:
# 몇개의 문장이 존재 하고 , 얼마나 줄었는지
len(result)

In [ ]:
# 형태소 분석을 통해 토큰화 진행된 상태이므로 word2vec 을 학습
from gensim.models import word2ved
model = word2vec(result, size = 100, window = 5, min_count = 5, workers = 4, sg = 0)

In [ ]:
model_result1 = model.wv.most_similar("대한민국")
print(model_result1)

In [ ]:
model_result2 = model.wv.most_similar("어벤져스")
print(model_result2)

In [ ]:
model_result3 = model.wv.most_similar("반도체")
print(model_result3)